# Data modeling 9: testing, metrics and change

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/data-modeling/data-modeling-09-testing-metrics-and-change.ipynb)

A model is a promise to the people who query it: this table has this grain, these keys resolve, this column
means net revenue after discounts. This lesson covers how to keep that promise: **tests** that fail the
pipeline when the data breaks the model, **documentation** stored with the tables, **metric definitions**
written once instead of in every dashboard, and **changing** a model without breaking its users.

**Prerequisites:** [Data modeling 2: star schema design](data-modeling-02-star-schema-design.ipynb). [Data modeling 3: slowly changing dimensions](data-modeling-03-slowly-changing-dimensions.ipynb) for the history tests.

*Last reviewed: November 2026, against DuckDB 2.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q duckdb~=2.0

## Setup and sample data

The next cell creates an in-memory DuckDB database and loads `src_customers` (the CRM's current view of each customer), `src_products` (the product catalogue), `src_orders` (one row per order, with fulfilment timestamps) and `src_order_lines` (one row per product on an order): extracts from a small online shop that sells coffee, tea and kitchenware. Every lesson starts from these source tables; [Data modeling 0](data-modeling-00-start-here.ipynb) describes them.

In [ ]:
import duckdb

con = duckdb.connect()  # an in-memory database: the warehouse for this notebook
con.sql("""
create table src_customers (
    customer_id int primary key, name varchar, email varchar, city varchar, segment varchar, created_on date
);
insert into src_customers values
    (101, 'Acme Corp',     'ops@acme.example',     'Leeds', 'smb',        '2025-11-02'),
    (102, 'Bright Retail', 'buy@bright.example',   'York',  'enterprise', '2025-11-20'),  -- was smb until Feb 1
    (103, 'Cyan Traders',  'hi@cyan.example',      'Bath',  'enterprise', '2025-12-05'),
    (104, 'Delta Grocers', 'orders@delta.example', 'Leeds', 'smb',        '2026-01-08'),  -- moved from Hull on Feb 15
    (105, 'Echo Supplies', null,                   'Derby', 'smb',        '2026-01-19'),
    (106, 'Foxtrot Foods', 'team@foxtrot.example', 'York',  'mid-market', '2026-02-10');  -- has not ordered yet
create table src_products (
    sku varchar primary key, product_name varchar, category varchar, department varchar, list_price decimal(8, 2)
);
insert into src_products values
    ('SKU-01', 'House Blend 1kg',    'Coffee',  'Grocery', 18.00),
    ('SKU-02', 'Espresso Beans 1kg', 'Coffee',  'Grocery', 22.00),
    ('SKU-03', 'Green Tea 100 bags', 'Tea',     'Grocery',  6.50),
    ('SKU-04', 'Earl Grey 100 bags', 'Tea',     'Grocery',  5.50),
    ('SKU-05', 'Stoneware Mug',      'Mugs',    'Kitchen',  9.00),
    ('SKU-06', 'Travel Mug',         'Mugs',    'Kitchen', 15.00),
    ('SKU-07', 'Gooseneck Kettle',   'Kettles', 'Kitchen', 65.00),
    ('SKU-08', 'Electric Kettle',    'Kettles', 'Kitchen', 45.00);
create table src_orders (
    order_id int primary key, customer_id int, order_ts timestamp, channel varchar, status varchar,
    shipping_fee decimal(8, 2), shipped_ts timestamp, delivered_ts timestamp
);
insert into src_orders values
    (1001, 101, '2026-01-05 09:12', 'web',   'delivered', 4.99, '2026-01-06 10:00', '2026-01-08 15:00'),
    (1002, 102, '2026-01-09 14:30', 'store', 'delivered', 0.00, '2026-01-09 14:30', '2026-01-09 14:30'),
    (1003, 103, '2026-01-14 11:05', 'web',   'delivered', 0.00, '2026-01-15 09:00', '2026-01-19 12:00'),
    (1004, 101, '2026-01-28 16:40', 'web',   'delivered', 4.99, '2026-01-29 11:00', '2026-02-02 10:30'),
    (1005, 104, '2026-02-03 10:20', 'web',   'delivered', 4.99, '2026-02-04 08:45', '2026-02-06 16:10'),
    (1006, 102, '2026-02-11 13:15', 'web',   'shipped',   0.00, '2026-02-12 09:30', null),
    (1007, 105, '2026-02-14 09:50', 'store', 'delivered', 0.00, '2026-02-14 09:50', '2026-02-14 09:50'),
    (1008, 104, '2026-02-20 18:05', 'web',   'placed',    4.99, null,               null),
    (1009, 103, '2026-02-24 12:00', 'web',   'cancelled', 4.99, null,               null),
    (1010, 101, '2026-02-27 08:30', 'store', 'delivered', 0.00, '2026-02-27 08:30', '2026-02-27 08:30');
create table src_order_lines (
    order_id int, line_no int, sku varchar, qty int, unit_price decimal(8, 2), discount decimal(8, 2),
    primary key (order_id, line_no)
);
insert into src_order_lines values
    (1001, 1, 'SKU-01',  2, 18.00,  0.00), (1001, 2, 'SKU-05', 4,  9.00, 0.00),
    (1002, 1, 'SKU-07',  1, 65.00,  5.00),
    (1003, 1, 'SKU-02', 10, 22.00, 20.00), (1003, 2, 'SKU-03', 5,  6.50, 0.00),
    (1004, 1, 'SKU-06',  2, 15.00,  0.00),
    (1005, 1, 'SKU-01',  3, 18.00,  0.00), (1005, 2, 'SKU-04', 2,  5.50, 0.00), (1005, 3, 'SKU-08', 1, 45.00, 0.00),
    (1006, 1, 'SKU-02', 20, 22.00, 40.00),
    (1007, 1, 'SKU-05',  6,  9.00,  0.00),
    (1008, 1, 'SKU-03',  4,  6.50,  0.00),
    (1009, 1, 'SKU-07',  1, 65.00,  0.00),
    (1010, 1, 'SKU-01',  1, 18.00,  0.00), (1010, 2, 'SKU-06', 1, 15.00, 0.00);
""")
con.sql("show tables").show()

The next cell builds a small star: `fact_sales` with `dim_customer` and `dim_product`, keyed by surrogate keys.

In [ ]:
con.sql("""
create table dim_customer as
select row_number() over (order by customer_id) as customer_sk, customer_id, name, segment from src_customers;
create table dim_product as
select row_number() over (order by sku) as product_sk, sku, product_name, category from src_products;
create table fact_sales as
select l.order_id, l.line_no, c.customer_sk, p.product_sk, cast(o.order_ts as date) as order_date, o.status,
       l.qty as quantity, l.qty * l.unit_price as gross_amount, l.discount as discount_amount,
       l.qty * l.unit_price - l.discount as net_amount
from src_order_lines l
join src_orders o using (order_id)
join dim_customer c using (customer_id)
join dim_product p using (sku)
""")
con.sql("select count(*) as fact_rows, sum(net_amount) as net_amount from fact_sales").show()

## Tests are queries that return the bad rows

Every modeling rule can be written as a query that returns the rows breaking it. An empty result means the rule
holds; any rows are the evidence. The core set for a dimensional model:

| Test | Rule | Catches |
|---|---|---|
| **unique** | the grain's key has no duplicates | double loads, fan-out in the build |
| **not null** | keys and required measures are present | broken joins, missing source fields |
| **relationships** | every fact key exists in its dimension | orphans that inner joins silently drop |
| **accepted values** | a code column holds only known values | new source statuses nobody mapped |
| **reconciliation** | the model's totals match the source's | rows lost or invented between layers |
| **freshness** | the newest row is recent enough | a stalled upstream feed |

Run them after each load and **fail the pipeline** when one returns rows, before anyone reads the bad data.
dbt's `unique`, `not_null`, `relationships` and `accepted_values` tests generate exactly these queries, and
packages such as dbt-expectations or tools such as Great Expectations and Soda add more.

**Where it matters**
- Every model, on every load; in CI against a sample before a model change is merged.

**Watch out for**
- Tests that warn instead of fail: warnings are read after the dashboard is wrong.
- Testing only row counts: a count can be right while keys are duplicated and others missing.

### Example

A small test suite for the star. `run_tests` raises an error if any test finds rows, like a pipeline's quality gate.

In [ ]:
TESTS = {
    "fact_sales grain is unique": "select order_id, line_no from fact_sales group by all having count(*) > 1",
    "fact_sales keys are not null": "select * from fact_sales where customer_sk is null or product_sk is null",
    "every customer_sk exists": "select f.* from fact_sales f anti join dim_customer d using (customer_sk)",
    "every product_sk exists": "select f.* from fact_sales f anti join dim_product d using (product_sk)",
    "status is a known value": "select distinct status from fact_sales where status not in ('placed', 'shipped', 'delivered', 'cancelled')",
    "net = gross - discount": "select * from fact_sales where net_amount <> gross_amount - discount_amount",
    "reconciles to source": """select * from (select (select sum(qty * unit_price - discount) from src_order_lines) as source_total,
                                                (select sum(net_amount) from fact_sales) as model_total)
                               where source_total <> model_total""",
}


def run_tests():
    failures = []
    for name, query in TESTS.items():
        bad_rows = con.sql(query).fetchall()
        print(f"{'FAIL' if bad_rows else 'pass'}  {name}" + (f": {len(bad_rows)} row(s), e.g. {tuple(map(str, bad_rows[0]))}" if bad_rows else ""))
        if bad_rows:
            failures.append(name)
    if failures:
        raise ValueError(f"{len(failures)} model test(s) failed: {failures}")


run_tests()

### Example

Now break the model the way real loads do: the source starts sending a new status, `returned`, and a retried
load inserts one line twice.

In [ ]:
con.sql("update fact_sales set status = 'returned' where order_id = 1004")
con.sql("insert into fact_sales select * from fact_sales where order_id = 1001 and line_no = 1")
try:
    run_tests()
except ValueError as error:
    print(f"\npipeline stopped: {error}")

**What to notice:** The clean model passes all seven tests. After the two faults, three fail: the duplicated line breaks the grain
and the reconciliation (the model has 1,118.50 against the source's 1,082.50, the duplicated line's 36.00), and the new status fails accepted values.
The relationship tests still pass, which is why a suite needs several kinds of test: each catches a different
failure. The error stops the pipeline before the duplicated revenue reaches a report.

## Document the model where it lives

The grain sentence, the meaning of each measure, and the rules applied (cancelled orders excluded, amounts after
line discounts, before shipping) belong with the table, not in a wiki that drifts. Warehouses store comments on
tables and columns (`comment on`), and catalogs and BI tools display them. dbt keeps the same descriptions in
YAML next to the model and its tests:

```yaml
models:
  - name: fact_sales
    description: "One row per product on an order (order line). Cancelled orders are included; filter on status."
    columns:
      - name: net_amount
        description: "Line revenue after line discounts, before shipping. Additive."
        tests: [not_null]
      - name: customer_sk
        tests:
          - relationships: {to: ref('dim_customer'), field: customer_sk}
```

Record for each measure whether it is additive, semi-additive or non-additive (lesson 1): a BI tool cannot tell.

**Where it matters**
- Every published table; the catalog (Unity Catalog, Glue, Dataplex) and BI semantic models read these comments.

**Watch out for**
- Descriptions that restate the column name ("net_amount: the net amount"): say the rule, the unit and the grain.

### Example

Store the grain and measure definitions as comments, then read them back from the catalog.

In [ ]:
con.sql("""
comment on table fact_sales is 'One row per product on an order (order line). Includes cancelled orders; filter on status.';
comment on column fact_sales.net_amount is 'Line revenue after line discounts, before shipping, in GBP. Additive.';
comment on column fact_sales.quantity is 'Units on the line. Additive.';
comment on column fact_sales.customer_sk is 'Surrogate key to dim_customer.';
""")
con.sql("select comment from duckdb_tables() where table_name = 'fact_sales'").show(max_width=200)
con.sql("""
select column_name, data_type, comment from duckdb_columns()
where table_name = 'fact_sales' and comment is not null order by column_index
""").show(max_width=200)

**What to notice:** The definitions are now part of the database: any tool that reads the catalog shows "Additive" and "before
shipping" next to the column, so the person building a chart does not have to guess. The columns without comments
are the gaps to close.

## Define each metric once

Most metric disputes are definition disputes. "Revenue" can be gross or net of discounts, with or without
cancelled orders, with or without shipping, by order date or payment date. When each dashboard writes its own
SQL, each picks differently, and three dashboards show three revenues.

A **metric definition** fixes the expression, the filters, the time dimension, the allowed dimensions and the
owner in one place. The simplest form is a governed view or table; a **semantic layer** (dbt's MetricFlow, Cube,
LookML, Snowflake semantic views) does the same and generates the SQL for any allowed grouping. Either way,
dashboards query the definition, not the fact table.

**Where it matters**
- Company KPIs, finance figures, anything shown to executives or customers.

**Watch out for**
- A metric layer that allows grouping by dimensions the metric does not support (a semi-additive balance summed over months).
- Ratios defined as averages of ratios instead of ratios of sums (lesson 1).

### Example

Two analysts compute February revenue. Then the team defines it once as a view.

In [ ]:
con.sql("""
select 'analyst A: gross, all orders'             as definition, sum(gross_amount) as february_revenue
from fact_sales where order_date between '2026-02-01' and '2026-02-28'
union all
select 'analyst B: net, excluding cancelled', sum(net_amount)
from fact_sales where order_date between '2026-02-01' and '2026-02-28' and status <> 'cancelled'
""").show()

In [ ]:
con.sql("""
create view metric_net_revenue as        -- owner: finance analytics. Net of line discounts, excludes cancelled orders,
select f.order_date,                     -- by order date, before shipping. Additive across every dimension below.
       c.segment, p.category,
       sum(f.net_amount) as net_revenue
from fact_sales f
join dim_customer c using (customer_sk)
join dim_product p using (product_sk)
where f.status <> 'cancelled'
group by all;
""")
con.sql("""
select date_trunc('month', order_date)::date as month, sum(net_revenue) as net_revenue
from metric_net_revenue group by all order by month
""").show()
con.sql("""
select segment, sum(net_revenue) as february_net_revenue
from metric_net_revenue where order_date between '2026-02-01' and '2026-02-28' group by all order by segment
""").show()

**What to notice:** The two ad-hoc definitions differ by more than 100 for the same month: analyst A included the cancelled order 1009
and counted discounts as revenue. With the view, the monthly total and the segment breakdown come from one
definition, so they add up: February's segment figures sum to the February total, because every report starts
from the same rule.

## Changing a model without breaking its users

Models change: a column is renamed, a type widens, a grain becomes finer, a dimension gains history. Some changes
are safe for consumers and some are not:

| Change | Safe? |
|---|---|
| add a nullable column, add a table | safe |
| widen a type (`int` to `bigint`, `decimal(10,2)` to `decimal(12,2)`) | usually safe |
| rename or drop a column, narrow a type | **breaking** |
| change a column's meaning (gross becomes net) | **breaking**, and silent |
| change the grain | **breaking** for every aggregate built on it |

For breaking changes, publish the new shape **alongside** the old: build `fact_sales_v2`, keep `fact_sales_v1`
(often a view over v2 that restores the old column names), announce a deprecation date, watch query logs until
nobody reads v1, then drop it. dbt calls these model versions. A **data contract** makes the published shape
explicit (column names, types, nullability, grain) and checks it on every build, so a breaking change fails in
CI instead of in a consumer's dashboard.

**Where it matters**
- Any table read by other teams, BI tools, reverse-ETL syncs or machine-learning features.

**Watch out for**
- Changing a column's meaning while keeping its name: no schema check notices, and every consumer is now wrong.

### Example

The team renames `net_amount` to `net_revenue` and drops `gross_amount`. Check the new table against the
published contract, then keep v1 consumers working with a view.

In [ ]:
CONTRACT_V1 = {  # the published interface of fact_sales
    "order_id": "INTEGER", "line_no": "INTEGER", "customer_sk": "BIGINT", "product_sk": "BIGINT",
    "order_date": "DATE", "status": "VARCHAR", "quantity": "INTEGER",
    "gross_amount": "DECIMAL(18,2)", "discount_amount": "DECIMAL(8,2)", "net_amount": "DECIMAL(19,2)",
}


def contract_violations(table, contract):
    actual = dict(con.sql(f"select column_name, column_type from (describe {table})").fetchall())
    problems = [f"missing column {c}" for c in contract if c not in actual]
    problems += [f"{c} changed type {contract[c]} -> {actual[c]}" for c in contract if c in actual and actual[c] != contract[c]]
    return problems


con.sql("""
create table fact_sales_v2 as
select order_id, line_no, customer_sk, product_sk, order_date, status, quantity, discount_amount,
       net_amount as net_revenue
from fact_sales
""")
print("fact_sales (v1):", contract_violations("fact_sales", CONTRACT_V1) or "matches the contract")
print("fact_sales_v2:  ", contract_violations("fact_sales_v2", CONTRACT_V1))

In [ ]:
con.sql("""
create view fact_sales_v1 as      -- the old interface, served from the new table during the deprecation window
select order_id, line_no, customer_sk, product_sk, order_date, status, quantity,
       cast(net_revenue + discount_amount as decimal(18, 2)) as gross_amount, discount_amount, net_revenue as net_amount
from fact_sales_v2
""")
print("fact_sales_v1 view:", contract_violations("fact_sales_v1", CONTRACT_V1) or "matches the contract")
con.sql("select order_id, line_no, gross_amount, net_amount from fact_sales_v1 where order_id = 1003 order by line_no").show()

**What to notice:** The contract check reports exactly what would break: `gross_amount` and `net_amount` are gone. Run in CI, that
blocks the change until the team decides how to release it. The `fact_sales_v1` view rebuilds the old columns from
the new table (gross is net plus discount), so existing dashboards keep working while they migrate to v2. The
view casts `gross_amount` back to `decimal(18,2)`: without the cast, the addition produces `decimal(20,2)`, and
the contract check flags that type change too.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** Your fact passes its row-count reconciliation but revenue is too high. Which test would catch it?

**A.** A uniqueness test on the grain key, and a reconciliation of totals (not just counts). A duplicated row and a missing row cancel out in a count but not in a sum.

**Q2.** Why should a failing test stop the pipeline instead of logging a warning?

**A.** Warnings are noticed after the bad data has been read. Stopping the load keeps the last good version in place and forces the problem to be fixed.

**Q3.** Two dashboards show different revenue for the same month. What do you look for first?

**A.** Different definitions: gross or net of discounts, cancelled orders included or not, shipping included or not, order date or payment date. Then define the metric once and point both at it.

**Q4.** Is changing `amount` from `decimal(10,2)` to `decimal(12,2)` a breaking change? Is changing it from gross to net?

**A.** Widening the type is usually safe. Changing the meaning is breaking even though the name and type stay the same, and no schema check will notice, so it needs a new column or version and an announcement.

**Q5.** How do you rename a column used by 40 dashboards?

**A.** Publish the new table or version with the new name, keep the old interface as a view mapping the new column to the old name, announce a deprecation date, watch query logs until the old name is unused, then drop the view.

## Key takeaways

- Write each modeling rule as a query that returns the rows breaking it, and fail the load when any test returns rows.
- Cover uniqueness, not-null keys, relationships, accepted values, reconciliation and freshness: each catches different faults.
- Store the grain and each measure's meaning and additivity as comments with the table.
- Define each metric once, as a governed view or in a semantic layer, and point dashboards at it.
- Additive schema changes are safe; renames, drops, meaning changes and grain changes need a new version beside the old.
- A data contract checked in CI turns a breaking change into a failed build instead of a broken dashboard.

**Next:** Continue with [Data modeling 10: interview case studies](data-modeling-10-interview-case-studies.ipynb).